In [ ]:
import os
import numpy as np
import scipy.stats as stats
import torch
import copy
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
#device = torch.device("cpu")

# Build the dataset

In [ ]:
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.nb_occ = {}

    def add_word(self, word):
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
            self.nb_occ[word] = 1
        else:
            self.nb_occ[word] += 1
        return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)

class CorpusWords:
    def __init__(self, path):
        self.dictionary = Dictionary()
        
        lst_seq, self.len_seq = self.tokenize(self.dictionary, path)
        pad_val = self.dictionary.word2idx["<empty>"]
        self.data = torch.nn.utils.rnn.pad_sequence(lst_seq, batch_first = True, padding_value = pad_val)
        self.max_len = max(self.len_seq)

    def tokenize(self, dct, path):
        """Tokenizes a text file."""
        assert os.path.exists(path)
        # Add words to the dictionary
        nb_sentences = 0
        with open(path, 'r', encoding="utf8") as f:
            for line in f:
                nb_sentences += 1
                words = ["<start>"] + line.split() + ["<end>"]
                for word in words:
                    self.dictionary.add_word(word)
        self.dictionary.add_word("<empty>")

        # Tokenize file content
        lst_seq = []
        len_seq = []
        with open(path, 'r', encoding="utf8") as f:
            for line in f:
                words = ["<start>"] + line.split() + ["<end>"]
                ln = len(words)
                ids = torch.LongTensor(ln)
                for i, word in enumerate(words):
                    ids[i] = self.dictionary.word2idx[word]
                lst_seq.append(ids)
                len_seq.append(ln)

        return lst_seq, torch.LongTensor(len_seq)

In [ ]:
corpus_fr = CorpusWords("small_vocab_fr.txt")
corpus_en = CorpusWords("small_vocab_en.txt")

In [ ]:
ntokens_enc = len(corpus_en.dictionary)
ntokens_dec = len(corpus_fr.dictionary)

pad_idx_en = corpus_en.dictionary.word2idx["<empty>"]
pad_idx_fr = corpus_fr.dictionary.word2idx["<empty>"] 

In [ ]:
batch_size = 128

dataset = torch.utils.data.TensorDataset(corpus_fr.data, corpus_en.data)
tvsets = torch.utils.data.random_split(dataset, [.9, .1])

train_set = tvsets[0]
test_set = tvsets[1]

train_loader = torch.utils.data.DataLoader(train_set, batch_size = batch_size, shuffle = True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size = batch_size, shuffle = False)

In [ ]:
def encode_sentence(sentence, dictionary, max_len):
    lst_words = ["<start>"] + sentence.split() + ["<end>"]
    l = len(lst_words)
    lst_words = lst_words + ["<empty>"] * (max_len - l)
    data = torch.empty(max_len, dtype = torch.int, device = device)
    for i, word in enumerate(lst_words):
        data[i] = dictionary.word2idx[word]
    return data, l

def decode_sentence(lst_idx, dictionary):
    lst_words = []
    for idx in lst_idx:
        lst_words.append(dictionary.idx2word[idx])
    return " ".join(lst_words)

# Transformer networks

A transformer network is made of several bricks.

On the *source* side:
* an embedding layer;
* a residual positional encoding layer;
* the encoder part of the transformer.

On the *target* side:
* an embedding layer;
* a residual positional encoding layer;
* the encoder part of the transformer.

The goal is to build such a neural network for automatic translation.

## Positional encoding

**Question 1**

Build the class `PositionalEncoding`, which implements the residual positoinal encoding layer.

Reminders:
1. positional encoding $\tilde{x} \in \mathbb{R}^{l \times d}$, where $l$ is the sequence length and $d$ the (even) dimension of the embedding:
$$
\tilde{x}_{p, 2 i} := \sin\left(\frac{p}{10000^{2i/d}}\right)
\qquad
\tilde{x}_{p, 2 i + 1} := \cos\left(\frac{p}{10000^{2i/d}}\right) ,
$$
where $p \in [0, l)$ and $i \in [0, d/2)$;
2. residual connection: $x \leftarrow x + \tilde{x}$;
3. drop-out layer with probability $p = 0.1$.

Useful tools: `torch.arange(...)`, `torch.outer(...)`, multiple indices accessors `[0::2]`...

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout = .1):
        super(PositionalEncoding, self).__init__()

        pass

    def forward(self, x):
        pass

## Masks

We want to perform automatic translation in the following way:
* we provide a *source* sentence to translate;
* we provide a *target_in* sentence, representing the part of the target sequence that has been already translated;
* we use a *target_out* sequence, representing the prediction goal.

In practice, *target_out* is *just target_in* shifted by one token: we use the first $n$ words of target_in to predict the $(n+1)$-th word of *target_in*, which is also the $n$-th word of *target_out*. The entire *source* sentence is used to make every prediction.

The `nn.Transformer` works in such a way that we can feed it with both *source* and *target_in* sentences to predict the words of *target_out*. However, it would be cheating to use the entire *target_in* sentence to predict *target_out*, because both sentences are identical, up to one shift. To prevent the Transformer from cheating, we have to use **masks**, which are boolean tensors indicating, for each prediction, which words the Transformer is allowed to use in *source* and *target_in* to perform each prediction on *target_out*.

For a sequence *target_in* of size $T$, the mask is a boolean tensor of size $T \times T$. If its $i$-th row of a mask is equal to `[False, False, False, True, True]`, then, to perform the $i$-th prediction, the Transformer is allowed to use the first 3 elements of the sequence *target_in*, but not the others.

For the *source*, we do not need a mask: we use the entire sequence to perform every prediction.

We will also use **padding masks** for *source* and *target_in*. These masks will be used to ignore the `<empty>` tokens in these sequences.

**Question 2**

Build a function `create_masks`, which takes the arguments:
* `src`: source tensor;
* `tgt`: target_in tensor;
* `pad_idx_src`: the index of `<empty>` in the source language;
* `pad_idx_tgt`: the index of `<empty>` in the target language;

and returns:
* `src_mask`: the source mask (normally, a matrix with `False` everywhere);
* `tgt_mask`: the target_in mask;
* `src_padding_mask`, `tgt_padding_mask`: masks indicating to ignore the tokens `<empty>` after the end of the sequences.

In [ ]:
def create_masks(src, tgt, pad_idx_src, pad_idx_tgt):
    pass

## The entire model

**Question 3**

Finish the class `ModelTransformer`, which will be our model. It inherits from `nn.Transformer`, which means that we will have to use attributes that are defined in this class, namely:
* `self.encoder`: the encoder part, which takes as inputs all the source-related objects (embedded input sentence, mask and padding_mask);
* `self.decoder`: the encoder part, which takes as inputs the output of the encoder (called `memory`) and all the target-related arguments (embedded target_in sentence, mask and padding_mask).

Do not forget the embeddings and the final fully-connected linear layer.

In [ ]:
class ModelTransformer(nn.Transformer):
    def __init__(self, ntokens_enc, ntokens_dec, max_len_src, max_len_tgt, batch_first = True,
                 d_model = 512, nhead = 8, dim_feedforward = 2048, nlayers = 6, dropout = .1):
        super(ModelTransformer, self).__init__(d_model = d_model, nhead = nhead, dim_feedforward = dim_feedforward, 
                                               num_encoder_layers = nlayers, dropout = dropout,
                                               batch_first = batch_first)
        # Note: self.encoder and self.decoder are already defined in the class nn.Transformer
        pass

    def forward(self, src, tgt, src_mask = None, tgt_mask = None,
               src_padding_mask = None, tgt_padding_mask = None):
        pass

## Training

**Question 4**

Finish the training and testing functions below.

You can show, at the end of every epoch, the source, the target and the ouput sentences, to visualize the progression of our model.

In [ ]:
def test_model(loader, model, criterion):
    model.eval()
    with torch.no_grad():
        train_loss = 0.
        valid_loss = 0.
        correct = 0
        total_pred = 0
        
        num_examples = 0
        for data_fr, data_en in loader:
            pass

        

def train_model(train_loader, test_loader, model, criterion, optimizer, nepochs):
    train_losses = []
    train_acc = []
    start_epoch = 0

    for epoch in range(nepochs):
        train_loss = 0.
        valid_loss = 0.
        correct = 0

        model.train()
        
        num_sentences = 0
        num_words = 0
        for data_fr, data_en in train_loader:
            pass

        # calculate average losses
        train_losses.append(train_loss)

        # print losses statistics
        
        print("=== Example ===")
        
        
        
        print("===============")

**Question 5**

Train a model and save it.

In [ ]:
d_model = 128
nhead = 4
dim_feedforward = 512
nlayers = 3
max_len_src = corpus_en.max_len
max_len_tgt = corpus_fr.max_len

model = ...

In [ ]:
criterion = nn.NLLLoss()

# optimizer
optimizer = ...

nepochs = 5

pass

## Testing

**Question 6**

Now that the model is trained, write the function `inference`, which translates a given sentence. It takes arguments:
* `model`: the model to use to perform the translation;
* `sentence`: the sentence to translate.

In [ ]:
def inference(model, sentence):
    pass